In [2]:
# Week 7 Lab Assignment
#This notebook contains two programs:
#1. Fourth-order Runge-Kutta method for a system of equations.
#2. Smallest eigenvalue using the power method.

In [3]:
import math

def add_vectors(a, b):
    return [a[i] + b[i] for i in range(len(a))]

def scale_vector(c, v):
    return [c * v[i] for i in range(len(v))]

def rk4_system(f, x0, y0, h, x_end):
    x = x0
    y = y0[:]
    results = [(x, y[:])]

    while x < x_end - 1e-12:
        k1 = scale_vector(h, f(x, y))
        k2 = scale_vector(h, f(x + h / 2, add_vectors(y, scale_vector(0.5, k1))))
        k3 = scale_vector(h, f(x + h / 2, add_vectors(y, scale_vector(0.5, k2))))
        k4 = scale_vector(h, f(x + h, add_vectors(y, k3)))

        for i in range(len(y)):
            y[i] = y[i] + (k1[i] + 2*k2[i] + 2*k3[i] + k4[i]) / 6

        x = round(x + h, 10)
        results.append((x, y[:]))

    return results


# Textbook test example for system of equations
# y1' = y1*e^(-x*y2) + 1
# y2' = y1*sin(x) - y2*cos(x)
# y1(0)=0, y2(0)=1
def system_example(x, y):
    y1 = y[0]
    y2 = y[1]

    dy1 = y1 * math.exp(-x * y2) + 1
    dy2 = y1 * math.sin(x) - y2 * math.cos(x)

    return [dy1, dy2]


x0 = 0.0
y0 = [0.0, 1.0]
h = 0.2
x_end = 2.0

results = rk4_system(system_example, x0, y0, h, x_end)

print("Fourth-Order Runge-Kutta Method for a System of Equations")
print()
print(f"{'x':>8} {'y1':>15} {'y2':>15}")
print("-" * 42)

for x, y in results:
    print(f"{x:8.2f} {y[0]:15.6f} {y[1]:15.6f}")

Fourth-Order Runge-Kutta Method for a System of Equations

       x              y1              y2
------------------------------------------
    0.00        0.000000        1.000000
    0.20        0.218956        0.822541
    0.40        0.473443        0.699310
    0.60        0.761383        0.642409
    0.80        1.077757        0.662686
    1.00        1.408498        0.770064
    1.20        1.729660        0.972518
    1.40        2.017985        1.275312
    1.60        2.266143        1.683197
    1.80        2.484759        2.205279
    2.00        2.689525        2.858736


In [4]:
#The first program uses the fourth-order Runge-Kutta method to solve a system of two differential equations. The output shows the values of y1 and y2 from x = 0 to x = 2

In [5]:
def dot(a, b):
    return sum(a[i] * b[i] for i in range(len(a)))

def mat_vec(A, x):
    result = []

    for i in range(len(A)):
        total = 0

        for j in range(len(A)):
            total += A[i][j] * x[j]

        result.append(total)

    return result

def norm_inf(v):
    return max(abs(value) for value in v)

def normalize(v):
    m = norm_inf(v)

    if m == 0:
        raise ValueError("Zero vector cannot be normalized.")

    return [value / m for value in v]

def solve_linear_system(A, b):
    n = len(A)
    M = [A[i][:] + [b[i]] for i in range(n)]

    for k in range(n):
        pivot_row = max(range(k, n), key=lambda r: abs(M[r][k]))

        if abs(M[pivot_row][k]) < 1e-15:
            raise ValueError("Matrix is singular.")

        M[k], M[pivot_row] = M[pivot_row], M[k]

        for i in range(k + 1, n):
            factor = M[i][k] / M[k][k]

            for j in range(k, n + 1):
                M[i][j] -= factor * M[k][j]

    x = [0.0] * n

    for i in range(n - 1, -1, -1):
        total = 0

        for j in range(i + 1, n):
            total += M[i][j] * x[j]

        x[i] = (M[i][n] - total) / M[i][i]

    return x

def inverse_power_smallest(A, x0, tolerance=1e-6, max_iterations=100):
    x = normalize(x0)
    old_lambda = None

    for iteration in range(1, max_iterations + 1):
        z = solve_linear_system(A, x)
        x = normalize(z)

        Ax = mat_vec(A, x)
        lambda_estimate = dot(x, Ax) / dot(x, x)

        if old_lambda is not None:
            error = abs(lambda_estimate - old_lambda)

            if error < tolerance:
                return lambda_estimate, x, iteration, error

        old_lambda = lambda_estimate

    return lambda_estimate, x, max_iterations, None


# Textbook test matrix
A = [
    [2.0, 1.0, 0.0, 0.0, 0.0],
    [1.0, 2.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 2.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 2.0, 1.0],
    [0.0, 0.0, 0.0, 1.0, 2.0]
]

x0 = [1.0, 1.0, 1.0, 1.0, 1.0]

eigenvalue, eigenvector, iterations, error = inverse_power_smallest(A, x0)

print("Smallest Eigenvalue using the Power Method")
print()
print("Matrix A:")

for row in A:
    print(row)

print()
print(f"Smallest eigenvalue = {eigenvalue:.6f}")
print("Eigenvector estimate =", [round(value, 6) for value in eigenvector])
print(f"Iterations = {iterations}")

if error is not None:
    print(f"Final error estimate = {error:.8f}")

Smallest Eigenvalue using the Power Method

Matrix A:
[2.0, 1.0, 0.0, 0.0, 0.0]
[1.0, 2.0, 1.0, 0.0, 0.0]
[0.0, 1.0, 2.0, 1.0, 0.0]
[0.0, 0.0, 1.0, 2.0, 1.0]
[0.0, 0.0, 0.0, 1.0, 2.0]

Smallest eigenvalue = 0.267949
Eigenvector estimate = [0.500032, -0.866041, 1.0, -0.866041, 0.500032]
Iterations = 6
Final error estimate = 0.00000005


In [6]:
#The second program finds the smallest eigenvalue using the inverse power method. Since the normal power method finds the largest eigenvalue, the inverse power method is used to find the smallest eigenvalue.